# Chapter 3: Coordinate Frames and Transformations

<a href="../lite/lab/index.html?path=ch03_coordinate_frames.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# ── Helper: build a 2D homogeneous transform ──────────────────────────────────
def make_transform(x, y, theta_rad):
    c, s = np.cos(theta_rad), np.sin(theta_rad)
    return np.array([[c, -s, x],
                     [s,  c, y],
                     [0,  0, 1]])

# ── Helper: draw a 2D coordinate frame ────────────────────────────────────────
def draw_frame(ax, T, label="", length=1.0, lw=2, fontsize=11):
    origin = T[:2, 2]
    x_axis = T[:2, 0] * length
    y_axis = T[:2, 1] * length
    ax.annotate("", xy=origin + x_axis, xytext=origin,
                arrowprops=dict(arrowstyle="->,head_width=0.25", color="red", lw=lw))
    ax.annotate("", xy=origin + y_axis, xytext=origin,
                arrowprops=dict(arrowstyle="->,head_width=0.25", color="blue", lw=lw))
    ax.plot(*origin, 'ko', ms=4)
    if label:
        ax.text(origin[0]-0.15, origin[1]-0.4, label, fontsize=fontsize,
                fontweight='bold', ha='center')

# ── Helper: draw a robot triangle ─────────────────────────────────────────────
def draw_robot(ax, T, size=0.6, color='steelblue', alpha=0.7):
    tri = np.array([[size, 0], [-size*0.5, size*0.5], [-size*0.5, -size*0.5], [size, 0]]).T
    tri_h = np.vstack([tri, np.ones((1, tri.shape[1]))])
    tri_w = (T @ tri_h)[:2]
    ax.fill(tri_w[0], tri_w[1], color=color, alpha=alpha)

# ── Helper: transform an array of 2D points ───────────────────────────────────
def transform_points(T, points):
    pts_h = np.vstack([points.T, np.ones(points.shape[0])])
    return (T @ pts_h)[:2].T

## 3.1 Points vs Vectors vs Frames

In robotics we constantly work with three distinct geometric objects:

| Object | What it represents | Depends on frame? |
|--------|-------------------|-------------------|
| **Point** | A location in space (e.g., a landmark at $(3, 5)$) | Yes — coordinates change when you change the frame |
| **Vector** | A displacement or direction (e.g., velocity $[1, 0]^T$) | Partially — magnitude is invariant, components depend on frame |
| **Frame** | An origin point **plus** an orientation (a coordinate system) | It *defines* the reference, so it's the anchor |

A **frame** $\{A\}$ is described by its origin ${}^{W}\mathbf{p}_A$ and a rotation ${}^{W}_{A}R$ relative
to some reference frame $\{W\}$.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
point = np.array([4.0, 3.0])       # a landmark position (meters)
velocity = np.array([1.5, 0.8])    # robot velocity vector (m/s)
frame_angle_deg = 30.0             # orientation of frame {B} (degrees)
frame_origin = np.array([2.0, 1.0])
# ──────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(1, 1, figsize=(8, 6))

# World frame
T_W = make_transform(0, 0, 0)
draw_frame(ax, T_W, label="{W}", length=1.2)

# Frame {B}
T_B = make_transform(*frame_origin, np.radians(frame_angle_deg))
draw_frame(ax, T_B, label="{B}", length=1.0)

# Point (landmark)
ax.plot(*point, 's', color='tomato', ms=12, zorder=5)
ax.annotate(f"  Point $P = ({point[0]:.0f}, {point[1]:.0f})$",
            xy=point, fontsize=11, color='tomato')

# Velocity vector (free vector — drawn from the point)
ax.annotate("", xy=point + velocity, xytext=point,
            arrowprops=dict(arrowstyle="->,head_width=0.3", color='forestgreen', lw=2.5))
ax.text(point[0] + velocity[0] + 0.15, point[1] + velocity[1],
        r"$\vec{v}$", fontsize=13, color='forestgreen', fontweight='bold')

ax.set_xlim(-1.5, 7); ax.set_ylim(-1, 6)
ax.set_aspect('equal')
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
ax.set_title("Points, Vectors, and Frames")
ax.legend(["landmark (point)"], loc="upper left")
plt.tight_layout()
plt.show()

## 3.2 Local vs Global Frames

A **LiDAR sensor** mounted on a robot returns range measurements in the **sensor's local frame**.
To build a map, we need every measurement expressed in the **global (world) frame**.

The transformation from local to global requires knowing the robot's **pose** — its position $(x, y)$
and heading $\theta$ in the world frame:

$${}^{W}\mathbf{p} = {}^{W}_{R}T \; {}^{R}\mathbf{p}$$

where ${}^{W}_{R}T$ is the 2D homogeneous transform from robot frame to world frame.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
robot_x = 3.0           # robot x in world frame (m)   (try 0, 5, -2)
robot_y = 2.0           # robot y in world frame (m)   (try 0, 3)
robot_theta_deg = 45.0  # robot heading (degrees)      (try 0, 90, 180)
n_scan_points = 40      # number of LiDAR rays
max_range = 5.0         # max sensor range (m)
wall_distance = 4.0     # distance to wall from origin (m)
# ──────────────────────────────────────────────────────────────────────────────

theta = np.radians(robot_theta_deg)
T_wr = make_transform(robot_x, robot_y, theta)

# Simulate a LiDAR scan: rays fan out from -90° to +90° in robot frame
angles_local = np.linspace(-np.pi/2, np.pi/2, n_scan_points)
# A wall at y = wall_distance in world frame → compute ranges in robot frame
scan_local = []
for a in angles_local:
    dx, dy = np.cos(a), np.sin(a)
    # Ray in world frame
    dx_w = np.cos(theta + a)
    dy_w = np.sin(theta + a)
    if dy_w > 0.01:
        r = (wall_distance - robot_y) / dy_w
        if 0 < r < max_range:
            scan_local.append([r * np.cos(a), r * np.sin(a)])
scan_local = np.array(scan_local) if scan_local else np.zeros((0, 2))

# Add some obstacle points (a box in local frame)
obs_angles = np.linspace(0.1, 0.5, 8)
obs_points = np.column_stack([2.0 * np.cos(obs_angles), 2.0 * np.sin(obs_angles)])
if scan_local.shape[0] > 0:
    scan_local = np.vstack([scan_local, obs_points])
else:
    scan_local = obs_points

# Transform to world frame
scan_world = transform_points(T_wr, scan_local)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: sensor frame
ax = axes[0]
ax.set_title("LiDAR scan — Sensor (local) frame", fontsize=13)
T_origin = make_transform(0, 0, 0)
draw_frame(ax, T_origin, label="{R}", length=1.0)
draw_robot(ax, T_origin, size=0.4)
ax.scatter(scan_local[:, 0], scan_local[:, 1], c='tomato', s=15, zorder=5, label='scan points')
for p in scan_local:
    ax.plot([0, p[0]], [0, p[1]], 'tomato', alpha=0.15, lw=0.5)
ax.set_xlim(-2, max_range+1); ax.set_ylim(-3, max_range+1)
ax.set_aspect('equal'); ax.set_xlabel("x_R (m)"); ax.set_ylabel("y_R (m)")
ax.legend()

# Right: world frame
ax = axes[1]
ax.set_title("LiDAR scan — World (global) frame", fontsize=13)
draw_frame(ax, make_transform(0, 0, 0), label="{W}", length=1.0)
draw_frame(ax, T_wr, label="{R}", length=0.8)
draw_robot(ax, T_wr, size=0.4)
ax.scatter(scan_world[:, 0], scan_world[:, 1], c='steelblue', s=15, zorder=5, label='scan (world)')
ax.axhline(y=wall_distance, color='gray', ls='--', lw=1.5, label='wall')
ax.set_xlim(-2, 10); ax.set_ylim(-2, wall_distance + 2)
ax.set_aspect('equal'); ax.set_xlabel("x_W (m)"); ax.set_ylabel("y_W (m)")
ax.legend()

plt.tight_layout()
plt.show()

**Key observations:**
- The same physical points look completely different depending on which frame you express them in.
- The robot's **pose** $(x, y, \theta)$ is the bridge between local and global descriptions.
- Every sensor reading must be transformed to a common frame before it can be fused into a map.

## 3.3 Rotation Matrices

A 2D **rotation matrix** maps vectors from one frame orientation to another:

$$R(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

**Properties:**
- **Orthogonal:** $R^T R = I$, so $R^{-1} = R^T$ (cheap to invert!)
- **Determinant:** $\det(R) = 1$ (preserves handedness)
- **Columns** are unit vectors: they are the axes of the rotated frame expressed in the original frame.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
rotation_deg = 40.0   # rotation angle (degrees)  (try 30, 90, -45, 180)
# ──────────────────────────────────────────────────────────────────────────────

theta = np.radians(rotation_deg)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

# A house shape
house = np.array([[0,0],[2,0],[2,1.5],[1,2.5],[0,1.5],[0,0]]).T
house_rotated = R @ house

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: the rotation
ax = axes[0]
ax.fill(house[0], house[1], color='steelblue', alpha=0.3, label='original')
ax.plot(house[0], house[1], 'steelblue', lw=2)
ax.fill(house_rotated[0], house_rotated[1], color='tomato', alpha=0.3, label=f'rotated {rotation_deg:.0f}°')
ax.plot(house_rotated[0], house_rotated[1], 'tomato', lw=2)
# Draw rotation matrix columns as vectors
ax.annotate("", xy=R[:, 0]*2, xytext=[0,0],
            arrowprops=dict(arrowstyle="->,head_width=0.2", color='red', lw=2))
ax.annotate("", xy=R[:, 1]*2, xytext=[0,0],
            arrowprops=dict(arrowstyle="->,head_width=0.2", color='blue', lw=2))
ax.text(R[0,0]*2.1, R[1,0]*2.1, r"$R\hat{x}$", color='red', fontsize=12)
ax.text(R[0,1]*2.1, R[1,1]*2.1, r"$R\hat{y}$", color='blue', fontsize=12)
ax.set_xlim(-3, 4); ax.set_ylim(-2, 4)
ax.set_aspect('equal'); ax.legend(fontsize=11)
ax.set_title(f"Rotation by {rotation_deg:.0f}°")

# Right: verify properties
ax = axes[1]
ax.axis('off')
RtR = R.T @ R
info = (f"R = \n{np.array2string(R, precision=3, suppress_small=True)}\n\n"
        f"R^T R = \n{np.array2string(RtR, precision=6, suppress_small=True)}\n\n"
        f"det(R) = {np.linalg.det(R):.6f}\n\n"
        f"R^{{-1}} = R^T? {np.allclose(np.linalg.inv(R), R.T)}")
ax.text(0.1, 0.5, info, fontsize=14, family='monospace',
        verticalalignment='center', transform=ax.transAxes,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax.set_title("Properties check")

plt.tight_layout()
plt.show()

**Key observations:**
- Columns of $R$ are where the original unit axes $\hat{x}$ and $\hat{y}$ end up after rotation.
- Because $R^{-1} = R^T$, inverting a rotation is just a transpose — extremely efficient.
- Rotation preserves lengths and angles (it is an **isometry**).

## 3.4 Translation and Rigid Transforms

A **rigid-body transform** combines rotation and translation:

$${}^{A}\mathbf{p} = R \; {}^{B}\mathbf{p} + \mathbf{t}$$

**Order matters!** Rotating then translating gives a different result than translating then rotating.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
tx, ty = 3.0, 1.0       # translation (meters)
angle_deg = 45.0         # rotation (degrees)
# ──────────────────────────────────────────────────────────────────────────────

theta = np.radians(angle_deg)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
t = np.array([tx, ty])

# L-shaped robot
shape = np.array([[0,0],[1.5,0],[1.5,0.5],[0.5,0.5],[0.5,1.2],[0,1.2],[0,0]]).T

# Route 1: rotate then translate
shape_rt = R @ shape + t[:, None]

# Route 2: translate then rotate
shape_tr = R @ (shape + t[:, None])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, transformed, title, order in [
    (axes[0], shape_rt, "Rotate THEN translate", "Rp + t"),
    (axes[1], shape_tr, "Translate THEN rotate", "R(p + t)")]:
    ax.fill(shape[0], shape[1], color='steelblue', alpha=0.3, label='original')
    ax.plot(shape[0], shape[1], 'steelblue', lw=2)
    ax.fill(transformed[0], transformed[1], color='tomato', alpha=0.3, label=order)
    ax.plot(transformed[0], transformed[1], 'tomato', lw=2)
    ax.set_xlim(-2, 6); ax.set_ylim(-2, 5)
    ax.set_aspect('equal'); ax.legend(fontsize=11); ax.set_title(title, fontsize=13)

plt.suptitle("Order matters: these are NOT the same transform!", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Key observations:**
- The standard convention in robotics is **rotate first, then translate**: ${}^{A}\mathbf{p} = R \; {}^{B}\mathbf{p} + \mathbf{t}$.
- This is exactly what the **homogeneous transform** matrix encodes (next section).
- Swapping the order produces a different result — a very common source of bugs.

## 3.5 Homogeneous Transformations

We pack rotation and translation into a single $3 \times 3$ matrix (for 2D):

$${}^{A}_{B}T = \begin{bmatrix} R & \mathbf{t} \\ \mathbf{0}^T & 1 \end{bmatrix} = \begin{bmatrix} \cos\theta & -\sin\theta & t_x \\ \sin\theta & \cos\theta & t_y \\ 0 & 0 & 1 \end{bmatrix}$$

Now we can **chain** transforms by matrix multiplication:

$${}^{A}_{C}T = {}^{A}_{B}T \; {}^{B}_{C}T$$

### Example: Two-link planar robot arm

A robot arm has a base frame, a shoulder joint, an elbow joint, and an end-effector.
Each joint adds a rotation and a link adds a translation along the local x-axis.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
link1_length = 3.0           # length of first link (m)    (try 2, 4)
link2_length = 2.0           # length of second link (m)   (try 1, 3)
joint1_angle_deg = 45.0      # shoulder angle (degrees)    (try 0, 30, 90, -45)
joint2_angle_deg = -30.0     # elbow angle (degrees)       (try 0, 45, -90)
# ──────────────────────────────────────────────────────────────────────────────

# Build transform chain: base → link1 → link2 → end-effector
T_base = make_transform(0, 0, 0)
T_01 = make_transform(0, 0, np.radians(joint1_angle_deg))                # shoulder rotation
T_12_translate = make_transform(link1_length, 0, 0)                       # along link 1
T_12_rotate = make_transform(0, 0, np.radians(joint2_angle_deg))          # elbow rotation
T_23_translate = make_transform(link2_length, 0, 0)                       # along link 2

# Compound transforms
T_0_joint1 = T_01
T_0_elbow = T_01 @ T_12_translate
T_0_joint2 = T_0_elbow @ T_12_rotate
T_0_ee = T_0_joint2 @ T_23_translate

fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Draw links
joints = [T_base[:2, 2], T_0_elbow[:2, 2], T_0_ee[:2, 2]]
for i in range(len(joints)-1):
    ax.plot([joints[i][0], joints[i+1][0]], [joints[i][1], joints[i+1][1]],
            'gray', lw=8, solid_capstyle='round', alpha=0.5)

# Draw frames
draw_frame(ax, T_base, "{Base}", length=0.8)
draw_frame(ax, T_0_elbow, "{Elbow}", length=0.6)
draw_frame(ax, T_0_ee, "{EE}", length=0.5)

# Mark joints
for j, label in zip(joints, ['base', 'elbow', 'EE']):
    ax.plot(*j, 'ko', ms=8, zorder=10)

# End-effector position
ee = T_0_ee[:2, 2]
ax.annotate(f"End-effector\n({ee[0]:.2f}, {ee[1]:.2f})", xy=ee,
            xytext=(ee[0]+0.5, ee[1]+0.8),
            arrowprops=dict(arrowstyle='->', color='tomato', lw=1.5),
            fontsize=12, color='tomato', fontweight='bold')

# Trace end-effector workspace (sweep joint2)
ee_trace = []
for a2 in np.linspace(-180, 180, 200):
    T = T_01 @ T_12_translate @ make_transform(0, 0, np.radians(a2)) @ T_23_translate
    ee_trace.append(T[:2, 2])
ee_trace = np.array(ee_trace)
ax.plot(ee_trace[:, 0], ee_trace[:, 1], 'orange', alpha=0.4, lw=1, label='elbow workspace')

lim = link1_length + link2_length + 1
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_aspect('equal')
ax.set_title(f"2-Link Robot Arm: θ₁={joint1_angle_deg:.0f}°, θ₂={joint2_angle_deg:.0f}°", fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

# Print the transforms
print("T_base→elbow (T_01 @ T_12_translate):")
print(np.array2string(T_0_elbow, precision=3, suppress_small=True))
print(f"\nT_base→EE (full chain):")
print(np.array2string(T_0_ee, precision=3, suppress_small=True))
print(f"\nEnd-effector position: ({ee[0]:.3f}, {ee[1]:.3f})")

**Key observations:**
- Each joint/link is one transform matrix. The full forward kinematics is just matrix multiplication.
- ${}^{0}_{EE}T = T_{01} \cdot T_{12} \cdot T_{23}$ — read left-to-right as "from base, go through joint 1, then link 1, then joint 2, …"
- The orange trace shows the **workspace** — all positions the end-effector can reach by sweeping one joint.

## 3.6 Composition and Inversion

**Composition** chains transforms: ${}^{A}_{C}T = {}^{A}_{B}T \; {}^{B}_{C}T$

**Inversion** reverses a transform. For a rigid-body transform, the efficient inverse is:

$${}^{B}_{A}T = \begin{bmatrix} R^T & -R^T \mathbf{t} \\ \mathbf{0}^T & 1 \end{bmatrix}$$

This avoids a general matrix inverse — we just transpose $R$ and adjust $\mathbf{t}$.

### Example: LiDAR → Robot → World pipeline

A LiDAR is mounted on a robot with a known offset. Points must pass through **two transforms**
to reach the world frame: sensor → robot → world.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
# Robot pose in world frame
robot_x, robot_y = 4.0, 3.0
robot_theta_deg = 30.0

# Sensor mounting on robot (offset from robot center)
sensor_dx = 0.5          # sensor is 0.5 m ahead of robot center
sensor_dy = 0.0
sensor_dtheta_deg = 0.0  # sensor aligned with robot (try 15, -10)
# ──────────────────────────────────────────────────────────────────────────────

T_wr = make_transform(robot_x, robot_y, np.radians(robot_theta_deg))     # world ← robot
T_rs = make_transform(sensor_dx, sensor_dy, np.radians(sensor_dtheta_deg))  # robot ← sensor
T_ws = T_wr @ T_rs  # world ← sensor (composition!)

# Simulated sensor readings (points in sensor frame)
np.random.seed(42)
n_pts = 30
# Wall points + a few cluster points
wall_pts = np.column_stack([np.random.uniform(2, 6, 20),
                            np.full(20, 3.0) + np.random.normal(0, 0.1, 20)])
obj_pts = np.column_stack([np.random.normal(1.5, 0.2, 10),
                           np.random.normal(1.0, 0.2, 10)])
scan_sensor = np.vstack([wall_pts, obj_pts])

# Transform through the pipeline
scan_robot = transform_points(T_rs, scan_sensor)
scan_world = transform_points(T_ws, scan_sensor)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Sensor frame
ax = axes[0]; ax.set_title("Sensor frame {S}", fontsize=13)
draw_frame(ax, make_transform(0,0,0), "{S}", length=0.8)
ax.scatter(scan_sensor[:,0], scan_sensor[:,1], c='tomato', s=15, zorder=5)
ax.set_xlim(-1, 7); ax.set_ylim(-2, 5); ax.set_aspect('equal')

# Robot frame
ax = axes[1]; ax.set_title("Robot frame {R}", fontsize=13)
draw_frame(ax, make_transform(0,0,0), "{R}", length=0.8)
draw_frame(ax, T_rs, "{S}", length=0.5)
draw_robot(ax, make_transform(0,0,0), size=0.3)
ax.scatter(scan_robot[:,0], scan_robot[:,1], c='orange', s=15, zorder=5)
ax.set_xlim(-1, 7); ax.set_ylim(-2, 5); ax.set_aspect('equal')

# World frame
ax = axes[2]; ax.set_title("World frame {W}", fontsize=13)
draw_frame(ax, make_transform(0,0,0), "{W}", length=1.0)
draw_frame(ax, T_wr, "{R}", length=0.6)
draw_frame(ax, T_ws, "{S}", length=0.4)
draw_robot(ax, T_wr, size=0.3)
ax.scatter(scan_world[:,0], scan_world[:,1], c='steelblue', s=15, zorder=5)
ax.set_xlim(-1, 12); ax.set_ylim(-1, 9); ax.set_aspect('equal')

plt.suptitle("Transform pipeline: Sensor → Robot → World", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Verify inversion
T_rw = np.eye(3)
T_rw[:2, :2] = T_wr[:2, :2].T
T_rw[:2, 2] = -T_wr[:2, :2].T @ T_wr[:2, 2]
print("T_wr @ T_rw (should be identity):")
print(np.array2string(T_wr @ T_rw, precision=6, suppress_small=True))

**Key observations:**
- Real robots have **multi-stage pipelines**: sensor → sensor mount → robot body → world.
- Each stage is one homogeneous transform; the full pipeline is their product.
- The efficient inverse $T^{-1}$ uses $R^T$ instead of a general matrix inverse.
- Getting one transform wrong (or in the wrong order) puts the entire map in the wrong place.

## 3.7 Debugging Frame Errors

The most common bugs in robotics code are **frame errors**:

1. **Wrong order of composition** — $T_{AB} \cdot T_{BC}$ vs $T_{BC} \cdot T_{AB}$
2. **Forgetting to invert** — using $T_{AB}$ where $T_{BA}$ is needed
3. **Wrong frame for a point** — treating a sensor-frame point as if it were in the world frame

These bugs are hard to catch numerically but **easy to spot visually**.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
robot_x, robot_y = 3.0, 2.0
robot_theta_deg = 40.0
show_bug = True    # True: show the wrong transform;  False: show only correct
# ──────────────────────────────────────────────────────────────────────────────

T_wr = make_transform(robot_x, robot_y, np.radians(robot_theta_deg))

# Some scan points in sensor frame
np.random.seed(7)
scan_local = np.column_stack([np.random.uniform(1, 4, 25),
                              np.random.uniform(-1.5, 1.5, 25)])

# Correct: world_points = T_wr @ sensor_points
scan_correct = transform_points(T_wr, scan_local)

# Bug 1: forgot to rotate (only translated)
scan_bug1 = scan_local + np.array([robot_x, robot_y])

# Bug 2: applied inverse instead of forward
T_rw = np.eye(3)
T_rw[:2, :2] = T_wr[:2, :2].T
T_rw[:2, 2] = -T_wr[:2, :2].T @ T_wr[:2, 2]
scan_bug2 = transform_points(T_rw, scan_local)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

titles = ["✓ Correct: T_wr @ p_sensor",
          "✗ Bug: forgot rotation (translate only)",
          "✗ Bug: used T_rw instead of T_wr"]
scans = [scan_correct, scan_bug1, scan_bug2]
colors = ['steelblue', 'tomato', 'tomato']

for ax, scan, title, c in zip(axes, scans, titles, colors):
    draw_frame(ax, make_transform(0,0,0), "{W}", length=0.8)
    draw_frame(ax, T_wr, "{R}", length=0.6)
    draw_robot(ax, T_wr, size=0.3)
    ax.scatter(scan[:,0], scan[:,1], c=c, s=15, zorder=5, alpha=0.8)
    if c == 'tomato' and show_bug:
        ax.scatter(scan_correct[:,0], scan_correct[:,1], c='steelblue', s=8,
                   zorder=4, alpha=0.3, label='correct (reference)')
        ax.legend(fontsize=9)
    ax.set_xlim(-4, 9); ax.set_ylim(-3, 7); ax.set_aspect('equal')
    ax.set_title(title, fontsize=11)

plt.suptitle("Frame bugs — can you spot what went wrong?", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Key observations:**
- **Always visualize** your transforms during development. Frame bugs are nearly invisible in numbers but obvious in plots.
- A sanity check: if the robot faces right and sees a wall ahead, the wall points in the world frame should be to the right of the robot.
- Use the subscript/superscript notation (${}^{A}_{B}T$) religiously — it makes composition rules mechanical.

---

## Exercises

### Exercise 3.1: Landmark in robot frame

A robot is at pose $(x, y, \theta) = (5, 3, 60°)$ in the world frame.
A landmark is at world coordinates $(8, 7)$.
Compute the landmark's position in the **robot's frame** (i.e., apply ${}^{R}_{W}T$).

In [ ]:
# Your code here
robot_pose = (5.0, 3.0, np.radians(60))
landmark_world = np.array([8.0, 7.0])

# Hint: build T_wr, invert it to get T_rw, then transform the landmark
# T_rw[:2,:2] = T_wr[:2,:2].T
# T_rw[:2, 2] = -T_wr[:2,:2].T @ T_wr[:2, 2]

### Exercise 3.2: Three-frame chain

Given:
- ${}^{A}_{B}T$: frame $\{B\}$ is at $(2, 1)$ with $30°$ rotation relative to $\{A\}$
- ${}^{B}_{C}T$: frame $\{C\}$ is at $(1, 0.5)$ with $-20°$ rotation relative to $\{B\}$

Compute ${}^{A}_{C}T$ and find the position of $\{C\}$'s origin in frame $\{A\}$.

In [ ]:
# Your code here
# T_ab = make_transform(2, 1, np.radians(30))
# T_bc = make_transform(1, 0.5, np.radians(-20))
# T_ac = ?

### Exercise 3.3: Multi-scan LiDAR registration

A robot takes two LiDAR scans from different poses:
- Scan 1 from pose $(1, 1, 0°)$
- Scan 2 from pose $(4, 2, 45°)$

Transform both scans to the world frame and plot them together.
Generate each scan as 20 random points in the sensor frame with x in $[1, 5]$ and y in $[-1, 1]$.

In [ ]:
# Your code here
np.random.seed(0)
scan1_local = np.column_stack([np.random.uniform(1, 5, 20), np.random.uniform(-1, 1, 20)])
scan2_local = np.column_stack([np.random.uniform(1, 5, 20), np.random.uniform(-1, 1, 20)])

# Hint: build T_w1 and T_w2, transform each scan, plot both on the same axes

### Exercise 3.4: Three-joint planar arm

Extend the robot arm example to **three links** with lengths $l_1 = 2$, $l_2 = 1.5$, $l_3 = 1$.
Compute the end-effector position for joint angles $\theta_1 = 30°$, $\theta_2 = 45°$, $\theta_3 = -60°$.
Plot the arm with all frames.

In [ ]:
# Your code here
l1, l2, l3 = 2.0, 1.5, 1.0
t1, t2, t3 = np.radians(30), np.radians(45), np.radians(-60)

# Hint: chain transforms like the 2-link example
# T = make_transform(0, 0, t1) @ make_transform(l1, 0, 0) @ ...

### Exercise 3.5: Relative transform between poses (challenge)

A robot observes the same landmark from two poses:
- From pose 1 $(0, 0, 0°)$, the landmark is at $(3, 2)$ in the sensor frame
- From pose 2, the landmark is at $(1, 3)$ in the sensor frame
- The landmark's world position (computed from pose 1) is $(3, 2)$

Given that the landmark is at the same world position in both observations,
estimate pose 2 (its position and orientation in the world frame).

*Hint: ${}^{W}\mathbf{p} = {}^{W}_{R_2}T \; {}^{R_2}\mathbf{p}$, so ${}^{W}_{R_2}T = ?$*

In [ ]:
# Your code here
# This is under-constrained with a single landmark in 2D.
# Use at least two landmarks to solve for pose 2.
# Landmarks in sensor frame 1:
p1_s1 = np.array([3, 2]); p2_s1 = np.array([5, 1])
# Same landmarks in sensor frame 2:
p1_s2 = np.array([1, 3]); p2_s2 = np.array([2.5, 2.8])
# Pose 1 is identity: T_w1 = I
# World positions of landmarks:
# p1_w = p1_s1, p2_w = p2_s1  (since T_w1 = I)
# Find T_w2 such that T_w2 @ p_s2 = p_w for both landmarks